In [2]:

import os
import requests
import pandas as pd
from datetime import datetime, timedelta
import pytz

# === Time Setup ===
eastern = pytz.timezone("US/Eastern")
yesterday = datetime.now(eastern) - timedelta(days=1)
date_str = yesterday.strftime('%Y-%m-%d')

# === File Output Setup ===
output_dir = os.path.abspath(os.path.join("..", "data", "game-scores"))
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, f"game_scores_{date_str}.csv")

# === Fetch Scores Function ===
def fetch_mlb_scores(target_date):
    url = f'https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={target_date.strftime("%Y-%m-%d")}'
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"❌ Failed to fetch data: {response.status_code}")
        return None

    data = response.json()
    games = data.get('dates', [])[0].get('games', []) if data.get('dates') else []

    game_data = []
    for game in games:
        home_team = game['teams']['home']['team']['name']
        away_team = game['teams']['away']['team']['name']

        home_score = game['teams']['home'].get('score', None)
        away_score = game['teams']['away'].get('score', None)

        game_info = {
            'Date': target_date.strftime('%Y-%m-%d'),
            'Home Team': home_team,
            'Home Score': home_score,
            'Away Team': away_team,
            'Away Score': away_score,
            'Status': game['status']['detailedState']
        }
        game_data.append(game_info)

    return pd.DataFrame(game_data)

# === Run ===
scores_df = fetch_mlb_scores(yesterday)

if scores_df is not None and not scores_df.empty:
    scores_df.to_csv(output_file, index=False)
    print(f"✅ Saved scores to {output_file}")
    print(scores_df)
else:
    print("⚠️ No scores found or failed to retrieve data.")


✅ Saved scores to /workspaces/MLB-Model/data/game-scores/game_scores_2025-06-18.csv
          Date             Home Team  Home Score              Away Team  \
0   2025-06-18      Seattle Mariners         1.0         Boston Red Sox   
1   2025-06-18         Miami Marlins         2.0  Philadelphia Phillies   
2   2025-06-18  Washington Nationals         1.0       Colorado Rockies   
3   2025-06-18      New York Yankees         2.0     Los Angeles Angels   
4   2025-06-18     Toronto Blue Jays         8.0   Arizona Diamondbacks   
5   2025-06-18       Cincinnati Reds         4.0        Minnesota Twins   
6   2025-06-18        Atlanta Braves         5.0          New York Mets   
7   2025-06-18        Tampa Bay Rays        12.0      Baltimore Orioles   
8   2025-06-18         Texas Rangers         3.0     Kansas City Royals   
9   2025-06-18  San Francisco Giants         2.0    Cleveland Guardians   
10  2025-06-18             Athletics         4.0         Houston Astros   
11  2025-06-18  

In [3]:
import os
import requests
import pandas as pd
from datetime import datetime
import pytz

# === Timestamp for file naming ===
eastern = pytz.timezone("US/Eastern")
today = datetime.now(eastern).date()
today_str = today.isoformat()

# === Where to save files ===
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))

def save_to_csv(df, subfolder, name):
    folder = os.path.join(BASE_DIR, subfolder)
    os.makedirs(folder, exist_ok=True)
    filepath = os.path.join(folder, f"{name}_{today_str}.csv")
    df.to_csv(filepath, index=False)
    print(f"✅ Saved: {filepath}")

# === Get official MLB team IDs ===
def get_mlb_teams():
    url = "https://statsapi.mlb.com/api/v1/teams?sportId=1"
    res = requests.get(url)
    res.raise_for_status()
    teams = res.json().get("teams", [])
    return {team["id"]: team["name"] for team in teams}

MLB_TEAMS = get_mlb_teams()

# === Fetch Team Stats (Pitching/Batting) ===
def fetch_team_stats(season="2025", group="pitching"):
    url = "https://statsapi.mlb.com/api/v1/teams/stats"
    params = {
        "season": season,
        "stats": "season",
        "group": group,
        "sportId": 1  # MLB only
    }
    res = requests.get(url, params=params)
    res.raise_for_status()
    data = res.json()

    rows = []
    for entry in data.get("stats", [])[0].get("splits", []):
        team_id = entry["team"]["id"]
        if team_id in MLB_TEAMS:
            stat = entry.get("stat", {})
            stat["team"] = MLB_TEAMS[team_id]
            rows.append(stat)

    return pd.DataFrame(rows)

# === Fetch MLB Standings ===
def fetch_standings(season="2025"):
    url = "https://statsapi.mlb.com/api/v1/standings"
    params = {
        "leagueId": "103,104",  # AL + NL
        "season": season,
        "standingsTypes": "regularSeason"
    }
    res = requests.get(url, params=params)
    res.raise_for_status()

    records = res.json().get("records", [])
    rows = []

    for division_record in records:
        division_name = division_record.get("division", {}).get("name", "Unknown")
        for team in division_record.get("teamRecords", []):
            split_records = team.get("records", {}).get("splitRecords", [])
            last_ten = next((r for r in split_records if r.get("type") == "lastTen"), {})
            rows.append({
                "team": team["team"]["name"],
                "division": division_name,
                "wins": team["wins"],
                "losses": team["losses"],
                "winPct": team["winningPercentage"],
                "gamesBack": team["gamesBack"],
                "runDiff": team.get("runDifferential"),
                "wildCardGB": team.get("wildCardGamesBack"),
                "lastTen": f"{last_ten.get('wins', '')}-{last_ten.get('losses', '')}"
            })

    return pd.DataFrame(rows)

# === Main Script ===
if __name__ == "__main__":
    try:
        # 1. Team Pitching
        print("📊 Fetching team pitching stats...")
        pitching_df = fetch_team_stats(group="pitching")
        save_to_csv(pitching_df, "team_pitching", "team_pitching")

        # 2. Team Batting
        print("📊 Fetching team batting stats...")
        batting_df = fetch_team_stats(group="hitting")
        save_to_csv(batting_df, "team_batting", "team_batting")

        # 3. Standings
        print("📊 Fetching league standings...")
        standings_df = fetch_standings()
        save_to_csv(standings_df, "standings", "standings")

        print("\n✅ All MLB team data fetched and saved successfully!")

    except Exception as e:
        print(f"❌ Error: {e}")


📊 Fetching team pitching stats...
✅ Saved: /workspaces/MLB-Model/data/team_pitching/team_pitching_2025-06-19.csv
📊 Fetching team batting stats...
✅ Saved: /workspaces/MLB-Model/data/team_batting/team_batting_2025-06-19.csv
📊 Fetching league standings...
✅ Saved: /workspaces/MLB-Model/data/standings/standings_2025-06-19.csv

✅ All MLB team data fetched and saved successfully!


In [4]:
import requests

url = "https://statsapi.mlb.com/api/v1/teams/stats"
params = {
    "season": "2025",
    "stats": "seasonAdvanced",
    "group": "hitting",
    "sportId": 1
}

response = requests.get(url, params=params)
response.raise_for_status()

# Preview keys in the first stat entry
splits = response.json().get("stats", [])[0].get("splits", [])
if splits:
    headers = list(splits[0].get("stat", {}).keys())
    print("✅ Available Advanced Hitting Stat Fields:")
    for h in headers:
        print("-", h)
else:
    print("⚠️ No data returned.")


✅ Available Advanced Hitting Stat Fields:
- plateAppearances
- totalBases
- leftOnBase
- sacBunts
- sacFlies
- babip
- extraBaseHits
- hitByPitch
- gidp
- gidpOpp
- numberOfPitches
- pitchesPerPlateAppearance
- walksPerPlateAppearance
- strikeoutsPerPlateAppearance
- homeRunsPerPlateAppearance
- walksPerStrikeout
- iso
- reachedOnError
- walkOffs
- flyOuts
- totalSwings
- swingAndMisses
- ballsInPlay
- popOuts
- lineOuts
- groundOuts
- flyHits
- popHits
- lineHits
- groundHits


In [8]:
import requests

# Base API paths
TEAM_STATS_URL = "https://statsapi.mlb.com/api/v1/teams/stats"
PLAYER_STATS_URL = "https://statsapi.mlb.com/api/v1/people/605151/stats"  # Example: Shohei Ohtani

# Configs to test all combos
scenarios = [
    {"level": "Team", "group": "hitting", "stats": "season"},
    {"level": "Team", "group": "hitting", "stats": "seasonAdvanced"},
    {"level": "Team", "group": "pitching", "stats": "season"},
    {"level": "Team", "group": "pitching", "stats": "seasonAdvanced"},
    {"level": "Player", "group": "hitting", "stats": "season"},
    {"level": "Player", "group": "hitting", "stats": "seasonAdvanced"},
    {"level": "Player", "group": "pitching", "stats": "season"},
    {"level": "Player", "group": "pitching", "stats": "seasonAdvanced"},
]

def print_stat_keys():
    for config in scenarios:
        print(f"\n📊 {config['level']} - {config['group'].capitalize()} - {config['stats']}")

        params = {
            "season": "2025",
            "group": config["group"],
            "stats": config["stats"],
            "sportId": 1
        }

        if config["level"] == "Team":
            res = requests.get(TEAM_STATS_URL, params=params)
        else:
            res = requests.get(PLAYER_STATS_URL, params=params)

        res.raise_for_status()
        stats = res.json().get("stats", [])

        if not stats:
            print(" ⚠️ No stats returned (empty list)")
            continue

        splits = stats[0].get("splits", [])
        if not splits:
            print(" ⚠️ No stat splits found")
            continue

        stat_keys = sorted(splits[0].get("stat", {}).keys())
        for key in stat_keys:
            print(" -", key)

print_stat_keys()


📊 Team - Hitting - season
 - airOuts
 - atBats
 - atBatsPerHomeRun
 - avg
 - babip
 - baseOnBalls
 - catchersInterference
 - caughtStealing
 - doubles
 - gamesPlayed
 - groundIntoDoublePlay
 - groundOuts
 - groundOutsToAirouts
 - hitByPitch
 - hits
 - homeRuns
 - intentionalWalks
 - leftOnBase
 - numberOfPitches
 - obp
 - ops
 - plateAppearances
 - rbi
 - runs
 - sacBunts
 - sacFlies
 - slg
 - stolenBasePercentage
 - stolenBases
 - strikeOuts
 - totalBases
 - triples

📊 Team - Hitting - seasonAdvanced
 - babip
 - ballsInPlay
 - extraBaseHits
 - flyHits
 - flyOuts
 - gidp
 - gidpOpp
 - groundHits
 - groundOuts
 - hitByPitch
 - homeRunsPerPlateAppearance
 - iso
 - leftOnBase
 - lineHits
 - lineOuts
 - numberOfPitches
 - pitchesPerPlateAppearance
 - plateAppearances
 - popHits
 - popOuts
 - reachedOnError
 - sacBunts
 - sacFlies
 - strikeoutsPerPlateAppearance
 - swingAndMisses
 - totalBases
 - totalSwings
 - walkOffs
 - walksPerPlateAppearance
 - walksPerStrikeout

📊 Team - Pitching - s